# ロケットシミュレーション

***
## 公式
#### D = Cd * Pv * A
#### 抵抗 = 抵抗係数 * 動圧力 * 面積

***
import

In [ ]:
from math import*
import numpy as np
import pandas as pd

: 

***
入力値

風向はロケットの進行方向を +　にする

In [ ]:
Cdx = float(input(" X Drag coefficient [ - ] : "))
Cdy = float(input(" Y Drag coefficient [ - ] : "))
Ax = float(input(" X area [ m^2 ] : "))
Ay = float(input(" Y area [ m^2 ] : "))
launch_deg = float(input(" launch Angle [ deg ] "))
wind_V = float(input(" Wind speed [ m/s ] : "))
wind_deg_yz = float(input(" WInd direction [ deg ] :"))
Tem0_C = float(input(" Surface Temperature [ C ] : "))
P0_hpa =float(input(" Sea level pressure [ hpa ] : "))
h0=float(input(" Altitude [ m ] : "))
mi=float(input(" weight [kg] : "))
mf=float(input(" weight final [ kg ] : "))
t_com=float(input(" Combustion time [ s ] : "))
t_act=float(input("Actuation time [ s ] : "))
T_com=float(input("Combustion Thrust [ N ] : "))
T_total=float(input("Total Thrust [ N ] : "))
k=float(input())

***
固定値

In [ ]:
L= 0.0065       
M= 0.028966
g0=9.80665
T_com_s=T_com/t_com   #연소시간 초당추력
T_act_s=(T_total-T_com)/(t_act-t_com) #작동시간 초당추력
launch_rad=launch_deg*pi/180
P0_pa=P0_hpa*100
Tem0_K=Tem0_C+273.15
wind_rad_yz = wind_deg_yz *pi / 180
wind_Vx = -wind_V * cos(wind_rad_yz) 
wind_Vy = -wind_V * sin(wind_rad_yz)
Δm_per_s=(mi-mf)/t_com

***
初期値

In [ ]:
m=mi
y=0
g=g0
Tem_K=Tem0_K
R=287
h=h0
atm= P0_pa 




ρ = atm * M / R / Tem_K


Vx=0
Vy=0
x=0
y=0
Pvx = 0.5 * ρ * pow((Vx+wind_Vx),2)
Pvy = 0.5 * ρ * pow((Vy+wind_Vy),2)

パラメータ

In [ ]:


Dx = Cdx * Pvx * Ax
Dy = Cdy * Pvy * Ay

In [ ]:
simulate_result={
    "t [s]" : [0],
    "x [m]" : [0],
    "y [m]" : [0],
    "vx" : [0],
    "vy" : [0]
}

***
# simulate

In [ ]:
for t in np.arange(0.0,100.0,k):

    if t<=t_com:
        T=T_com_s
    elif t_com<t<=t_act:
        T=T_act_s
    else:
        T=0
    
    if t<=t_com:
        m=mi-Δm_per_s*k
    else:
        m=mf

    Dx = Cdx * Pvx * Ax
    Dy = Cdy * Pvy * Ay

    Pvx = 0.5 * ρ * pow((Vx+wind_Vx),2)
    Pvy = 0.5 * ρ * pow((Vy+wind_Vy),2)
    h=h0+y
    atm= P0_pa * pow((1-(L*h)/Tem0_K),(g*M/R/L))
    g=pow((6371000),2)/pow((6371000+h),2)*g0
    Tem_K=Tem0_K-0.6*0.001*h
    ρ = atm * M / R / Tem_K
    R=atm*(22.4+Tem_K-273.15/273.15)

    #加速度
    ax= (T*cos(launch_rad)-Dx)/m
    if t>=t_act:
        ay=-9.8
    else:
        ay= (T*sin(launch_rad)-Dy)/m-g

    #速度
    Vx=Vx+ax*k
    Vy=Vy+ay*k

    #位置
    x=x+Vx*k
    y=y+Vy*k

    #配列化
    simulate_result["t [s]"].append(t)
    simulate_result["x [m]"].append(x)
    simulate_result["y [m]"].append(y)
    simulate_result["vx"].append(Vx)
    simulate_result["vy"].append(Vy)

    if simulate_result["y [m]"][-1]<simulate_result["y [m]"][-2]:
        break

    

In [ ]:
data=pd.DataFrame(simulate_result)
data.to_csv("simulate_result_data_test.csv",encoding='utf-8-sig',index=False)